In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from qaoa_rw import data, qubo, qaoa, baselines, metrics

RESULTS, FIGS = pathlib.Path("../results"), pathlib.Path("../figures")
RESULTS.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
FAST = False
N_QUBITS, K = 12, 5
P_MAX = 3 if FAST else 4
RESTARTS = 3 if FAST else 10

X, y, names, rel, red = data.load_problem(N_QUBITS, seed=SEED)
Q, offset = qubo.feature_selection_qubo(rel, red, K, 0.5, 2.0)
E = qubo.energies(Q, offset)
opt = int(np.argmin(E))
print("E_min =", round(E[opt], 4), "| E_media =", round(E.mean(), 4))


In [ ]:
import time
runs, prev = {}, None
for p in range(1, P_MAX + 1):
    t0 = time.time()
    inits = [qaoa.interp_init(prev["params"])] if prev is not None else None
    r = qaoa.run_qaoa(E, N_QUBITS, p, restarts=RESTARTS if p == 1 else 4, seed=SEED + p, x0_list=inits, maxiter=1000)
    runs[p], prev = r, r
    print(f"p={p}: <E>={r['expectation']:.4f}  nfev={r['nfev']}  ({time.time()-t0:.1f}s)")


In [ ]:
rows = []
for p, r in runs.items():
    rows.append({"método": f"QAOA p={p}", **metrics.summarize(E, r["prob"], N_QUBITS, K)})

g = baselines.greedy(Q, offset, N_QUBITS, K)
sa = baselines.simulated_annealing(Q, offset, N_QUBITS, seed=SEED)
x_opt = qubo.index_to_mask(opt, N_QUBITS)
classical = {"Greedy": g, "Simulated annealing": sa, "Óptimo exacto": x_opt}
for name, x in classical.items():
    e = float(x @ Q @ x + offset)
    rows.append({"método": name, "energía_esperada": e, "razón_aprox": metrics.approx_ratio(e, E),
                 "prob_óptimo": np.nan, "prob_factible": float(x.sum() == K)})
summary = pd.DataFrame(rows).set_index("método")
summary.round(4)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ps = list(runs)
ax[0].plot(ps, [summary.loc[f"QAOA p={p}", "razón_aprox"] for p in ps], "o-", color="#2a6f97", label="QAOA")
for name, c in [("Greedy", "#e07a5f"), ("Simulated annealing", "#81b29a")]:
    ax[0].axhline(summary.loc[name, "razón_aprox"], ls="--", color=c, label=name)
ax[0].axhline(1, color="k", lw=.8); ax[0].set_xticks(ps); ax[0].set(xlabel="p", ylabel="razón de aproximación", title="Calidad vs profundidad"); ax[0].legend()

ax[1].plot(ps, [summary.loc[f"QAOA p={p}", "prob_factible"] for p in ps], "o-", color="#2a6f97")
ax[1].axhline(metrics.feasible_mask(N_QUBITS, K).mean(), ls="--", color="gray", label="azar")
ax[1].set_xticks(ps); ax[1].set(xlabel="p", ylabel="P(solución factible)", title="Factibilidad"); ax[1].legend()

order = np.argsort(E)
for p in (1, P_MAX):
    ax[2].plot(np.cumsum(runs[p]["prob"][order]), label=f"QAOA p={p}")
ax[2].plot(np.linspace(0, 1, len(E)), "--", color="gray", label="azar")
ax[2].set(xlabel="bitstrings ordenados por energía", ylabel="probabilidad acumulada", title="¿Dónde pone la masa QAOA?"); ax[2].legend()
plt.tight_layout(); plt.savefig(FIGS / "02_qaoa_sim.png", dpi=150); plt.show()


In [ ]:
rng = np.random.default_rng(SEED)
shots_grid = [10, 30, 100, 300, 1000]
uniform = np.full(len(E), 1 / len(E))
curves = {"Aleatorio": uniform, **{f"QAOA p={p}": runs[p]["prob"] for p in (1, P_MAX)}}
best = {lab: [metrics.expected_best_energy(pr, E, s, 200, rng) for s in shots_grid] for lab, pr in curves.items()}

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for lab, v in best.items():
    ax.plot(shots_grid, v, "o-", label=lab)
ax.axhline(E.min(), color="k", lw=.8, ls="--", label="óptimo")
ax.set(xscale="log", xlabel="disparos N", ylabel="energía del mejor de N", title="Mejor de N disparos"); ax.legend()
plt.tight_layout(); plt.savefig(FIGS / "02_best_of_n.png", dpi=150); plt.show()
pd.DataFrame(best, index=shots_grid).round(3)


In [ ]:
rng = np.random.default_rng(SEED)
acc = {"Las 12 candidatas": metrics.cv_accuracy(X, y, np.ones(N_QUBITS))}
for p in (1, P_MAX):
    i = metrics.best_sample_index(runs[p]["prob"], E, 1000, rng)
    acc[f"QAOA p={p} (mejor de 1000)"] = metrics.cv_accuracy(X, y, qubo.index_to_mask(i, N_QUBITS))
for name, x in classical.items():
    acc[name] = metrics.cv_accuracy(X, y, x)
pd.Series(acc, name="accuracy CV (5 pliegues)").round(4).to_frame()


In [ ]:
summary.to_csv(RESULTS / "02_sim_summary.csv")
json.dump({str(p): {"params": r["params"].tolist(), "expectation": r["expectation"]} for p, r in runs.items()},
          open(RESULTS / f"02_params_n{N_QUBITS}_k{K}.json", "w"), indent=1)
print("Resultados guardados en", RESULTS)
